In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from transformers import TimesformerModel, TimesformerConfig

config = TimesformerConfig.from_pretrained("facebook/timesformer-base-finetuned-k400")
model = TimesformerModel.from_pretrained("facebook/timesformer-base-finetuned-k400", config=config)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

In [4]:
print(config)


TimesformerConfig {
  "architectures": [
    "TimesformerForVideoClassification"
  ],
  "attention_probs_dropout_prob": 0.0,
  "attention_type": "divided_space_time",
  "drop_path_rate": 0,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "id2label": {
    "0": "abseiling",
    "1": "air drumming",
    "2": "answering questions",
    "3": "applauding",
    "4": "applying cream",
    "5": "archery",
    "6": "arm wrestling",
    "7": "arranging flowers",
    "8": "assembling computer",
    "9": "auctioning",
    "10": "baby waking up",
    "11": "baking cookies",
    "12": "balloon blowing",
    "13": "bandaging",
    "14": "barbequing",
    "15": "bartending",
    "16": "beatboxing",
    "17": "bee keeping",
    "18": "belly dancing",
    "19": "bench pressing",
    "20": "bending back",
    "21": "bending metal",
    "22": "biking through snow",
    "23": "blasting sand",
    "24": "blowing glass",
    "25": "blowing leaves",
    "26": "blowing nose",
    

In [5]:
import os
import cv2
import torch
import numpy as np
from torchvision import transforms
from transformers import TimesformerModel, TimesformerConfig

In [6]:
# Paths
video_folder = "/content/drive/MyDrive/video_captioning/video_sample"
output_feature_folder = "/content/drive/MyDrive/video_captioning/msvd_features"

os.makedirs(output_feature_folder, exist_ok=True)

In [7]:
# Load pretrained TimeSformer model & config (frozen)
model_name = "facebook/timesformer-base-finetuned-k400"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading pretrained TimeSformer...")
config = TimesformerConfig.from_pretrained(model_name)
model = TimesformerModel.from_pretrained(model_name, config=config)
model.eval()
model.to(device)

Loading pretrained TimeSformer...


model.safetensors:   0%|          | 0.00/486M [00:00<?, ?B/s]

TimesformerModel(
  (embeddings): TimesformerEmbeddings(
    (patch_embeddings): TimesformerPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (time_drop): Dropout(p=0.0, inplace=False)
  )
  (encoder): TimesformerEncoder(
    (layer): ModuleList(
      (0-11): 12 x TimesformerLayer(
        (drop_path): Identity()
        (attention): TimeSformerAttention(
          (attention): TimesformerSelfAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
          )
          (output): TimesformerSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): TimesformerIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (dropout): Dropout(p=0.0, inpla

In [8]:
# ImageNet normalization stats
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

In [9]:
def sample_frames(video_path, num_frames=8):
    """Sample num_frames uniformly from the video."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < num_frames:
        # If fewer frames, duplicate last frame to pad
        frame_idxs = list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)
    else:
        frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if i in frame_idxs:
            frames.append(frame)
    cap.release()
    return frames

In [10]:
def preprocess_frames(frames):
    """Apply preprocessing to frames and stack into tensor."""
    processed = [preprocess(frame) for frame in frames]  # list of [3,224,224]
    video_tensor = torch.stack(processed, dim=0)  # shape [T,3, H, W]
    return video_tensor.unsqueeze(0)  # [1,T,3, H, W]


Testing for single video

In [11]:
# def extract_features(video_id):
#     video_path = os.path.join(video_folder, f"{video_id}.mp4")  # adjust extension if needed
#     if not os.path.exists(video_path):
#         print(f"Video not found: {video_path}")
#         return None

#     frames = sample_frames(video_path, num_frames=config.num_frames)
#     input_tensor = preprocess_frames(frames).to(device)

#     with torch.no_grad():
#         outputs = model(input_tensor)

#     # outputs.last_hidden_state shape: [batch_size, sequence_length, hidden_size]
#     # We can take the [CLS] token embedding at position 0
#     cls_embedding = outputs.last_hidden_state[:, 0, :]  # take  cls token [1, hidden_size]

#     return cls_embedding.squeeze(0).cpu()  # [hidden_size]





# # Example usage
# video_ids = ["video0"]

# for vid in video_ids:
#     features = extract_features(vid)
#     if features is not None:
#         save_path = os.path.join(output_feature_folder, f"{vid}.pt")
#         torch.save(features, save_path)
#         print(f"Saved features for {vid} to {save_path}")


For full video folder

In [12]:
def extract_and_save_features(video_id):
    video_path = os.path.join(video_folder, f"{video_id}.avi")  # Adjust extension if needed
    if not os.path.exists(video_path):
        print(f"Video not found: {video_path}")
        return

    feature_path = os.path.join(output_feature_folder, f"{video_id}.pt")
    if os.path.exists(feature_path):
        print(f"Features already exist for {video_id}, skipping.")
        return

    frames = sample_frames(video_path, num_frames=config.num_frames)
    input_tensor = preprocess_frames(frames).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token embedding

    torch.save(cls_embedding.squeeze(0).cpu(), feature_path)
    print(f"Saved features for {video_id} to {feature_path}")

# List all video IDs by scanning your video folder
video_files = [f for f in os.listdir(video_folder) if f.endswith(".avi")]
video_ids = [os.path.splitext(f)[0] for f in video_files]

# Loop over all videos and extract features
for vid in video_ids:
    extract_and_save_features(vid)

Features already exist for 00661_6owu8Mow0_g_527_534, skipping.
Features already exist for 00154_WqwcWk1In_g_46_54, skipping.
Features already exist for 01007_5OuYhq6Zl0g_0_10, skipping.
Features already exist for 00405_9QI8cgBSGo8_28_41, skipping.
Features already exist for 00437_io2dbV-Qbus_215_247, skipping.
Features already exist for 00260_97JhYpoWxzY_0_4, skipping.
Features already exist for 00725_3FnUTQMJVXI_31_36, skipping.
Features already exist for 00221_PiyoeFC31kE_9_27, skipping.
Features already exist for 00693_lvFYUmDSOvU_34_38, skipping.
Features already exist for 01016_WTf5EgVY5uU_100_104, skipping.
Features already exist for 00668_hSgGBHbJrmE_0_17, skipping.
Features already exist for 00043_xxHx6s_DbUo_173_177, skipping.
Features already exist for 00399_2YhDTpzxd3c_98_101, skipping.
Features already exist for 00316_FOOM-wA2rOY_77_86, skipping.
Features already exist for 00720_xxHx6s_DbUo_121_128, skipping.
Features already exist for 00020_ScdUht-pM6s_53_63, skipping.
Fe

In [13]:
import torch

features = torch.load('/content/drive/MyDrive/video_captioning/msvd_features/00178_xaPepCVepCg_35_46.pt')
print(features.shape)
print(features)


torch.Size([768])
tensor([-5.5383e-01,  7.6765e-01, -3.5736e-01,  2.5639e-01,  3.8794e-01,
         1.1066e-01,  2.7103e-01,  1.2491e-01,  8.6486e-01,  4.8470e-01,
        -1.1142e+00,  1.8995e-01, -1.2038e+00, -4.3071e-01, -7.3999e-01,
         5.8095e-01, -2.1211e-01, -1.0975e+00, -1.1489e+00, -5.2815e-01,
        -1.1710e+00, -5.7582e-01, -6.0483e-02, -8.9018e-01,  6.2209e-01,
        -2.3866e+00,  6.7640e-02,  1.3266e+00,  3.9276e-01, -3.6261e-01,
        -2.2047e-01, -7.6999e-01, -2.1131e-01, -8.8593e-01,  8.5661e-01,
        -1.2399e-02, -2.9293e-01,  6.9999e-01, -1.9585e+00, -1.1647e-01,
         5.0844e-01,  7.5824e-02, -1.3955e+00, -1.4717e-01,  1.5510e+00,
        -1.1722e-01,  1.7759e-01,  1.5784e+00,  2.9008e-01,  6.2095e-01,
        -1.5252e+00, -3.4008e-01,  1.2526e+00,  3.8563e-01,  2.6911e+00,
        -8.2084e-01, -6.4861e-02, -1.3272e-01, -1.4332e+00,  9.1827e-02,
         4.5805e-01,  1.6082e-01,  1.5265e-01,  8.5902e-01,  1.7923e-01,
         1.7216e+00,  6.9173e-01,

MAking the dataset to train the adapter


In [14]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import MBart50TokenizerFast

class VideoCaptionDataset(Dataset):
    def __init__(self, csv_path, features_dir, tokenizer_name="facebook/mbart-large-50-many-to-many-mmt"):
        """
        Args:
            csv_path (str): Path to the CSV file containing 'video_id' and 'nepali_caption'
            features_dir (str): Directory where .pt files of video embeddings are stored
            tokenizer_name (str): Name of the pretrained tokenizer
        """
        self.data = pd.read_csv(csv_path)
        self.features_dir = features_dir

        # Initialize tokenizer
        self.tokenizer = MBart50TokenizerFast.from_pretrained(tokenizer_name)
        self.tokenizer.src_lang = "ne_NP"

        # === Calculate 95th percentile of tokenized caption lengths ===
        lengths = []
        for caption in self.data['nepali_caption']:
            tokens = self.tokenizer(caption, return_tensors="pt", truncation=False)
            lengths.append(tokens["input_ids"].shape[1])
        self.max_length = int(np.percentile(lengths, 95))

        print(f"[INFO] 95th percentile of caption lengths: {self.max_length}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        video_id = row['video_id']
        caption = row['nepali_caption']

        # === 1. Load video embedding ===
        feature_path = os.path.join(self.features_dir, f"{video_id}.pt")
        video_embedding = torch.load(feature_path)  # [1, seq_len, 1024]
        video_embedding = video_embedding.squeeze(0)  # [seq_len, 1024]

        # === 2. Tokenize caption ===
        tokens = self.tokenizer(
            caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        # === 3. Return input ===
        return {
            "video_embedding": video_embedding,                        # [seq_len, 1024]
            "input_ids": tokens["input_ids"].squeeze(0),              # [max_length]
            "attention_mask": tokens["attention_mask"].squeeze(0)     # [max_length]
        }


In [15]:
from torch.utils.data import DataLoader


dataset = VideoCaptionDataset(
    csv_path="/content/drive/MyDrive/video_captioning/train.csv",
    features_dir="/content/drive/MyDrive/video_captioning/msvd_features",
)

tokenizer = dataset.tokenizer


# Create DataLoader
batch_size = 8  # Adjust based on your GPU memory
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,        # Shuffle dataset for training
    num_workers=4,       # Number of parallel workers for loading (adjust based on your CPU)
    pin_memory=True      # Speeds up transfer to GPU
)


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[INFO] 95th percentile of caption lengths: 20


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [16]:
# Example: Load one sample
sample = dataset[0]
print("Embedding shape:", sample["video_embedding"].shape)       # [seq_len, 1024]
print("Input IDs shape:", sample["input_ids"].shape)             # [20]
print("Attention mask shape:", sample["attention_mask"].shape)   # [20]
print(dataset[7])


Embedding shape: torch.Size([768])
Input IDs shape: torch.Size([20])
Attention mask shape: torch.Size([20])
{'video_embedding': tensor([ 1.1036e+00, -4.4228e-01, -8.9027e-01,  2.9341e-01, -5.6963e-01,
         5.4514e-01,  1.0231e+00, -9.5418e-02,  4.4592e-02,  3.1134e-01,
        -6.4908e-01, -4.4302e-01, -4.6457e-01, -6.3358e-01, -6.4399e-01,
         9.8424e-01, -1.1604e-01, -4.8612e-02,  3.4271e-01, -2.5586e+00,
        -2.7922e+00,  1.5358e+00, -1.5331e-02,  3.3096e-01, -1.8899e-01,
        -9.5935e-01, -1.8756e-01, -7.0935e-02, -3.2080e-01, -5.3573e-02,
        -1.7578e-01,  7.3769e-01,  4.1078e-01, -4.6674e-01, -2.6475e-01,
        -8.2466e-01, -7.9467e-01,  6.9038e-01, -1.0298e+00,  3.2425e-01,
         6.4649e-01, -2.5375e-02,  1.3227e+00, -7.1551e-01,  1.1765e+00,
         2.4171e+00, -1.4423e+00, -1.0651e-01,  7.2029e-01,  3.9307e-01,
         1.1936e+00, -2.0987e-01, -9.8982e-01, -8.3302e-01, -4.4928e-01,
         7.6909e-01,  6.2184e-02,  4.2935e-01,  1.8076e+00,  5.1517e-

Adapter Model


In [17]:
import torch
import torch.nn as nn

class VideoToTextAdapter(nn.Module):
    def __init__(self, input_dim=768, output_dim=1024, seq_len=10):
        super(VideoToTextAdapter, self).__init__()
        self.seq_len = seq_len

        # Linear layer to project TimeSformer embedding to mBART hidden size
        self.linear = nn.Linear(input_dim, output_dim)

        # Learnable positional embeddings to give each "pseudo-token" a position
        self.pos_emb = nn.Parameter(torch.randn(seq_len, output_dim))

    def forward(self, video_feat):
        """
        video_feat: [batch_size, 768] — one embedding per video
        returns: [batch_size, seq_len, 1024] — token-like sequence for mBART
        """
        projected = self.linear(video_feat)  # [batch, 1024]

        # Repeat the same projected vector seq_len times
        expanded = projected.unsqueeze(1).repeat(1, self.seq_len, 1)  # [batch, seq_len, 1024]

        # Add positional embeddings
        pseudo_tokens = expanded + self.pos_emb.unsqueeze(0)  # [batch, seq_len, 1024]

        return pseudo_tokens


In [18]:
#checking adapter by dummy input

adapter = VideoToTextAdapter(input_dim=768, output_dim=1024, seq_len=10)
dummy_input = torch.randn(4, 768)  # batch_size=4
output = adapter(dummy_input)
print(output.shape)  # Should print: torch.Size([4, 10, 1024])


torch.Size([4, 10, 1024])


Loading Mbart decoder

In [19]:
from transformers import MBartForConditionalGeneration
# Load the pretrained mBART model
model = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')

# Freeze the encoder parameters to avoid training them
for param in model.model.encoder.parameters():
    param.requires_grad = False

# Now only decoder and adapter parameters will be trainable
print("Encoder frozen:", all(not p.requires_grad for p in model.model.encoder.parameters()))
print("Decoder trainable:", any(p.requires_grad for p in model.model.decoder.parameters()))

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Encoder frozen: True
Decoder trainable: True


Training LOop

In [20]:
from tqdm import tqdm
import torch
from torch.optim import AdamW
from transformers.modeling_outputs import BaseModelOutput

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
adapter.to(device)

model.train()
adapter.train()

optimizer = AdamW(list(adapter.parameters()) + list(model.model.decoder.parameters()), lr=5e-5)

num_epochs = 5

for epoch in range(num_epochs):
    total_loss = 0

    # Wrap dataloader with tqdm
    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)

    for batch in loop:
        video_embedding = batch['video_embedding'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        encoder_hidden_states = adapter(video_embedding)
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

        decoder_input_ids = input_ids[:, :-1]
        labels = input_ids[:, 1:].clone()
        labels[labels == tokenizer.pad_token_id] = -100

        outputs = model(
            encoder_outputs=encoder_outputs,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=attention_mask[:, :-1],
            labels=labels,
            use_cache=False
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm progress bar with loss
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")


Epoch 1/5:   1%|          | 65/6141 [01:38<2:32:53,  1.51s/it, loss=5.18]


KeyboardInterrupt: 

Testing the video classification to see if the model is working properly or not  

In [ ]:
# import os
# import cv2
# import torch
# import numpy as np
# from torchvision import transforms
# from transformers import TimesformerForVideoClassification, TimesformerConfig
# import requests

# # --- Paths ---
# video_folder = "/content/drive/MyDrive/video_captioning"
# video_filename = "video19.mp4"  # change to your video file
# video_path = os.path.join(video_folder, video_filename)

# # --- Load Kinetics-400 Labels ---
# # Download kinetics labels file (if not present)
# labels_url = "https://raw.githubusercontent.com/deepmind/kinetics-i3d/master/data/label_map.txt"
# labels_path = "kinetics_labels.txt"
# if not os.path.exists(labels_path):
#     r = requests.get(labels_url)
#     with open(labels_path, "w") as f:
#         f.write(r.text)
# with open(labels_path, "r") as f:
#     kinetics_labels = [line.strip() for line in f.readlines()]

# # --- Device ---
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # --- Model ---
# model_name = "facebook/timesformer-base-finetuned-k400"
# print("Loading Timesformer classification model...")
# model = TimesformerForVideoClassification.from_pretrained(model_name)
# model.eval()
# model.to(device)

# # --- Preprocessing ---
# imagenet_mean = [0.485, 0.456, 0.406]
# imagenet_std = [0.229, 0.224, 0.225]

# preprocess = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
# ])

# def sample_frames(video_path, num_frames=8):
#     cap = cv2.VideoCapture(video_path)
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#     if total_frames < num_frames:
#         frame_idxs = list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)
#     else:
#         frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

#     frames = []
#     for i in range(total_frames):
#         ret, frame = cap.read()
#         if not ret:
#             break
#         if i in frame_idxs:
#             frames.append(frame)
#     cap.release()
#     return frames

# def preprocess_frames(frames):
#     processed = [preprocess(frame) for frame in frames]  # list of [3,224,224]
#     video_tensor = torch.stack(processed, dim=0)  # [T,3,H,W]
#     return video_tensor.unsqueeze(0)  # [1,T,3,H,W]

# # --- Run Classification ---
# frames = sample_frames(video_path, num_frames=8)
# input_tensor = preprocess_frames(frames).to(device)

# with torch.no_grad():
#     outputs = model(input_tensor)
#     logits = outputs.logits  # [1, 400]
#     probs = torch.nn.functional.softmax(logits, dim=-1)
#     top5_prob, top5_catid = torch.topk(probs, 5)

# print("Top 5 Kinetics-400 predictions:")
# for i in range(top5_prob.size(1)):
#     print(f"{kinetics_labels[top5_catid[0,i]]}: {top5_prob[0,i].item()*100:.2f}%")
